# Tokenize News Articles, Clean the Text and Analyse the Most Frequent Terms

**Aim:** To tokenize and preprocess news articles by removing unwanted elements, applying stemming and
lemmatization, and finding the most frequent terms.

**Dataset:** AG News (120,000 training and 7,600 test articles, 4 categories).

**Steps:** lowercase → clean → tokenize (words, sentences, characters) → remove stop words → stem / lemmatize →
count frequent terms → **test the pipeline** (unit tests, unseen test articles, per-category terms).

In [ ]:
!pip install -q datasets nltk

In [ ]:
import re
import string
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from datasets import load_dataset

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

## 1. Load the dataset

In [ ]:
# [LOAD]
dataset = load_dataset("sh0416/ag_news")

# Convert to pandas
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

label_names = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

print("Training samples:", len(train_df))
print("Testing samples :", len(test_df))
print("\nColumns:", list(train_df.columns))

print("\nArticles per category (training set):")
print(train_df["label"].map(label_names).value_counts())

print("\nFirst 5 articles:")
display(train_df.head())

In [ ]:
# Combine title and description into one text column
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

print(train_df["text"].iloc[0])

## 2. Normalise: lowercase

In [ ]:
def lowercase_text(text):
    return text.lower()

train_df["lowercase_text"] = train_df["text"].apply(lowercase_text)

print("Original :", train_df["text"].iloc[0])
print("Lowercase:", train_df["lowercase_text"].iloc[0])

## 3. Remove punctuation and special characters

In [ ]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)

print(train_df["clean_text"].iloc[0])

## 4. Tokenization: words, sentences and characters

In [ ]:
def tokenize_words(text):
    return word_tokenize(text)

train_df["word_tokens"] = train_df["clean_text"].apply(tokenize_words)

print("Word tokens:")
print(train_df["word_tokens"].iloc[0])

In [ ]:
# Sentence tokenization (done on the original text, because it needs the punctuation)
sample_text = train_df["text"].iloc[0]
sentences = sent_tokenize(sample_text)

print("Number of sentences:", len(sentences))
for i, sentence in enumerate(sentences[:5], 1):
    print(f"{i}. {sentence}")

# A longer example with several sentences
multi = "Oil prices rose on Monday. Analysts said demand is strong! Will the trend continue? Nobody knows."
print("\nSentences found in a multi-sentence text:")
for i, s in enumerate(sent_tokenize(multi), 1):
    print(f"{i}. {s}")

In [ ]:
# Character tokenization
def tokenize_characters(text):
    return list(text)

chars = tokenize_characters("Oil prices soar")
print("Characters:", chars)
print("Number of characters:", len(chars))

## 5. Remove stop words

In [ ]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

train_df["filtered_tokens"] = train_df["word_tokens"].apply(remove_stopwords)

print("Before removing stopwords:")
print(train_df["word_tokens"].iloc[0])

print("\nAfter removing stopwords:")
print(train_df["filtered_tokens"].iloc[0])

## 6. Stemming and lemmatization

In [ ]:
stemmer = PorterStemmer()

def stem_words(tokens):
    return [stemmer.stem(word) for word in tokens]

train_df["stemmed_tokens"] = train_df["filtered_tokens"].apply(stem_words)

print("Original tokens:")
print(train_df["filtered_tokens"].iloc[0])

print("\nStemmed tokens:")
print(train_df["stemmed_tokens"].iloc[0])

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

train_df["lemmatized_tokens"] = train_df["filtered_tokens"].apply(lemmatize_words)

print("Original tokens:")
print(train_df["filtered_tokens"].iloc[0])

print("\nLemmatized tokens:")
print(train_df["lemmatized_tokens"].iloc[0])

## 7. Complete preprocessing function

In [ ]:
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. Tokenization
    tokens = word_tokenize(text)

    # 5. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

train_df["processed_tokens"] = train_df["text"].apply(preprocess_text)

print(train_df["processed_tokens"].iloc[0])

## 8. Most frequent terms

In [ ]:
# Combine all processed tokens
all_words = []
for tokens in train_df["processed_tokens"]:
    all_words.extend(tokens)

word_frequency = Counter(all_words)
most_common_words = word_frequency.most_common(20)

print("20 Most Frequent Terms:\n")
for word, frequency in most_common_words:
    print(f"{word}: {frequency}")

In [ ]:
frequency_df = pd.DataFrame(most_common_words, columns=["Term", "Frequency"])
display(frequency_df)

In [ ]:
plt.figure(figsize=(12, 6))
words = [item[0] for item in most_common_words]
frequencies = [item[1] for item in most_common_words]

plt.bar(words, frequencies)
plt.xlabel("Terms")
plt.ylabel("Frequency")
plt.title("Top 20 Most Frequent Terms in AG News")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
all_stems = []
for tokens in train_df["stemmed_tokens"]:
    all_stems.extend(tokens)

stem_frequency = Counter(all_stems)

print("Top 20 Stemmed Terms:")
for word, frequency in stem_frequency.most_common(20):
    print(f"{word}: {frequency}")

In [ ]:
all_lemmas = []
for tokens in train_df["lemmatized_tokens"]:
    all_lemmas.extend(tokens)

lemma_frequency = Counter(all_lemmas)

print("Top 20 Lemmatized Terms:")
for word, frequency in lemma_frequency.most_common(20):
    print(f"{word}: {frequency}")

In [ ]:
# Vocabulary size at every stage: how much do stemming / lemmatization shrink the vocabulary?
raw_vocab   = set(w for t in train_df["word_tokens"] for w in t)
stop_vocab  = set(w for t in train_df["filtered_tokens"] for w in t)
stem_vocab  = set(all_stems)
lemma_vocab = set(all_lemmas)

vocab_df = pd.DataFrame({
    "Stage": ["After tokenization", "After stop-word removal", "After stemming", "After lemmatization"],
    "Unique terms": [len(raw_vocab), len(stop_vocab), len(stem_vocab), len(lemma_vocab)],
})
vocab_df["Reduction vs. tokenization"] = (
    (1 - vocab_df["Unique terms"] / len(raw_vocab)) * 100
).round(1).astype(str) + " %"
display(vocab_df)

In [ ]:
index = 0

print("========== ORIGINAL ==========")
print(train_df["text"].iloc[index])
print("\n========== LOWERCASE ==========")
print(train_df["lowercase_text"].iloc[index])
print("\n========== CLEANED ==========")
print(train_df["clean_text"].iloc[index])
print("\n========== TOKENS ==========")
print(train_df["word_tokens"].iloc[index])
print("\n========== STOPWORDS REMOVED ==========")
print(train_df["filtered_tokens"].iloc[index])
print("\n========== STEMMED ==========")
print(train_df["stemmed_tokens"].iloc[index])
print("\n========== LEMMATIZED ==========")
print(train_df["lemmatized_tokens"].iloc[index])

---
# Testing the preprocessing pipeline

Three kinds of tests are done:
1. **Unit tests** - every function is checked against an answer we know.
2. **Unseen data test** - the pipeline is run on the 7,600 *test* articles that were never used above.
3. **Sanity test** - the most frequent terms of every news category should make sense.

## Test 1: unit tests for every preprocessing function

In [ ]:
def run_test(name, actual, expected):
    passed = actual == expected
    results.append({"Test": name, "Result": "PASS" if passed else "FAIL",
                    "Expected": str(expected), "Actual": str(actual)})

results = []

run_test("Lowercase",
         lowercase_text("Oil PRICES Soar"),
         "oil prices soar")

run_test("Remove punctuation / special characters",
         clean_text("Wall St. Bears Claw Back!! (Reuters) - $5bn #deal"),
         "wall st bears claw back reuters bn deal")

run_test("Remove digits",
         clean_text("2004 was the 28th Olympics"),
         "was the th olympics")

run_test("Remove extra spaces",
         clean_text("  too    many     spaces  "),
         "too many spaces")

run_test("Word tokenization",
         tokenize_words("oil prices soar"),
         ["oil", "prices", "soar"])

run_test("Sentence tokenization (3 sentences)",
         len(sent_tokenize("Prices rose. Stocks fell! Is it over?")),
         3)

run_test("Character tokenization",
         tokenize_characters("abc"),
         ["a", "b", "c"])

run_test("Stop-word removal",
         remove_stopwords(["the", "oil", "is", "in", "the", "tank"]),
         ["oil", "tank"])

run_test("Stemming (Porter)",
         stem_words(["running", "companies", "prices", "played"]),
         ["run", "compani", "price", "play"])

run_test("Lemmatization (WordNet, nouns)",
         lemmatize_words(["bears", "companies", "prices", "geese"]),
         ["bear", "company", "price", "goose"])

run_test("Full pipeline on a sentence",
         preprocess_text("The Oil Prices are SOARING, analysts said!"),
         ["oil", "prices", "soaring", "analysts", "said"])

run_test("Full pipeline on empty text",
         preprocess_text(""),
         [])

test_report = pd.DataFrame(results)
display(test_report)

passed = (test_report["Result"] == "PASS").sum()
print(f"\nUnit tests passed: {passed} / {len(test_report)}")

## Test 2: run the pipeline on unseen test articles

In [ ]:
test_df["processed_tokens"] = test_df["text"].apply(preprocess_text)
test_df["stemmed_tokens"]   = test_df["processed_tokens"].apply(stem_words)

raw_len  = test_df["text"].apply(lambda t: len(word_tokenize(t)))
kept_len = test_df["processed_tokens"].apply(len)

train_vocab = set(all_words)
test_words  = [w for tokens in test_df["processed_tokens"] for w in tokens]
test_vocab  = set(test_words)
oov_types   = test_vocab - train_vocab
oov_tokens  = sum(1 for w in test_words if w not in train_vocab)

summary = pd.DataFrame({
    "Measure": [
        "Test articles processed",
        "Average tokens per article before cleaning",
        "Average tokens per article after cleaning",
        "Tokens removed (punctuation + stop words)",
        "Articles that became empty",
        "Unique terms in test set",
        "Terms never seen in training (OOV types)",
        "OOV rate (share of test tokens)",
    ],
    "Value": [
        len(test_df),
        round(raw_len.mean(), 1),
        round(kept_len.mean(), 1),
        f"{(1 - kept_len.sum() / raw_len.sum()) * 100:.1f} %",
        int((kept_len == 0).sum()),
        len(test_vocab),
        len(oov_types),
        f"{oov_tokens / len(test_words) * 100:.2f} %",
    ],
})
display(summary)

assert (kept_len == 0).sum() == 0, "some articles became empty after preprocessing"
print("Check passed: no article is empty after preprocessing.")

In [ ]:
# Do the training and test sets agree on the most frequent terms?
train_top = [w for w, _ in word_frequency.most_common(20)]
test_top  = [w for w, _ in Counter(test_words).most_common(20)]
overlap   = sorted(set(train_top) & set(test_top))

print(f"Top-20 terms shared by training and test data: {len(overlap)} / 20")
print(overlap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, (title, counter) in zip(axes, [("Training set", Counter(all_words)), ("Test set", Counter(test_words))]):
    top = counter.most_common(10)
    ax.barh([w for w, _ in top][::-1], [c for _, c in top][::-1])
    ax.set_title(f"Top 10 terms - {title}")
    ax.set_xlabel("Frequency")
plt.tight_layout()
plt.show()

## Test 3: sanity check - frequent terms of every category

In [ ]:
rows = []
for label, name in label_names.items():
    cat_tokens = [w for tokens in test_df.loc[test_df["label"] == label, "processed_tokens"] for w in tokens]
    top = Counter(cat_tokens).most_common(8)
    rows.append({"Category": name, "Articles": int((test_df["label"] == label).sum()),
                 "Top terms": ", ".join(f"{w} ({c})" for w, c in top)})

display(pd.DataFrame(rows))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (label, name) in zip(axes.ravel(), label_names.items()):
    cat_tokens = [w for tokens in test_df.loc[test_df["label"] == label, "processed_tokens"] for w in tokens]
    top = Counter(cat_tokens).most_common(8)
    ax.barh([w for w, _ in top][::-1], [c for _, c in top][::-1])
    ax.set_title(name)
plt.suptitle("Most frequent terms per news category (test set)")
plt.tight_layout()
plt.show()

## Test 4: try the pipeline on your own headline

In [ ]:
def show_pipeline(text):
    lowered   = lowercase_text(text)
    cleaned   = clean_text(text)
    tokens    = tokenize_words(cleaned)
    filtered  = remove_stopwords(tokens)
    stemmed   = stem_words(filtered)
    lemmatized = lemmatize_words(filtered)
    steps = [
        ("Original", text), ("Lowercase", lowered), ("Cleaned", cleaned),
        ("Tokens", tokens), ("Stop words removed", filtered),
        ("Stemmed", stemmed), ("Lemmatized", lemmatized),
    ]
    display(pd.DataFrame(steps, columns=["Step", "Output"]))

show_pipeline("Apple Inc. shares jumped 5% on Tuesday after the company's new iPhones were launched in the U.S.!")

## Conclusion

The news articles were converted to lowercase, cleaned, tokenized into words, sentences and characters, stripped of
stop words and reduced with stemming and lemmatization. The most frequent terms were counted for every stage.
The testing section showed that every function gives the expected answer, that the pipeline runs on unseen articles
without producing empty results, and that the frequent terms of each category (Sports, Business, ...) are meaningful.